<a href="https://colab.research.google.com/github/ankita-rdk/ALZ-MRI-Classification/blob/main/alz1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
files.upload()

Saving alz_kaggle.json to alz_kaggle.json


{'alz_kaggle.json': b'{"username":"ankitasos","key":"a5a23a0a200aa0a96aa96b5f52e9c3ff"}'}

In [2]:
# Setup Kaggle API key
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download the 44,000 image balanced dataset
!kaggle datasets download -d aryansinghal10/alzheimers-multiclass-dataset-equal-and-augmented

# Unzip silently (-q) and overwrite (-o) without prompting
!unzip -o -q alzheimers-multiclass-dataset-equal-and-augmented.zip

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/aryansinghal10/alzheimers-multiclass-dataset-equal-and-augmented
License(s): apache-2.0
100% 398M/398M [00:11<00:00, 36.5MB/s]



In [3]:
!unzip -o -q alzheimers-multiclass-dataset-equal-and-augmented.zip

In [4]:
# CELL 1: SETUP & DATA PIPELINE
import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# 1. AUTO-DETECT THE COLAB DATASET PATH
DATA_DIR = None
print("Searching for dataset in Colab...")
for root, dirs, files in os.walk('./'):
    if 'MildDemented' in dirs and 'NonDemented' in dirs:
        DATA_DIR = root
        break

if DATA_DIR is None:
    raise FileNotFoundError("Could not find the dataset! Make sure the unzip command finished.")

print(f"✅ Success! Dataset found at: {DATA_DIR}")

# 2. CONFIGURATION
BATCH_SIZE = 64
IMAGE_SIZE = 224
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {device}")

# 3. TRANSFORMS
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 4. LOAD & SPLIT (80/20)
full_dataset = datasets.ImageFolder(root=DATA_DIR)
class_names = full_dataset.classes
print(f"Detected Classes: {class_names}")

total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size = total_size - train_size
train_subset, val_subset = random_split(full_dataset, [train_size, val_size])

class DatasetWrapper(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

train_dataset = DatasetWrapper(train_subset, transform=train_transform)
val_dataset = DatasetWrapper(val_subset, transform=val_transform)

# Using num_workers=2 for Colab's hardware limits
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Training samples: {len(train_dataset)} | Validation samples: {len(val_dataset)}")

Searching for dataset in Colab...
✅ Success! Dataset found at: ./combined_images
Using Device: cpu
Detected Classes: ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
Training samples: 35200 | Validation samples: 8800


In [5]:
# CELL 2: STATE-OF-THE-ART DENSENET121 ARCHITECTURE
class DenseNetAlzheimer(nn.Module):
    def __init__(self, num_classes):
        super(DenseNetAlzheimer, self).__init__()

        # 1. Load the pre-trained DenseNet121 (The Gold Standard for Medical Imaging)
        weights = models.DenseNet121_Weights.DEFAULT
        self.base_model = models.densenet121(weights=weights)

        # 2. Freeze the foundational layers
        # DenseNet121 has dense blocks. We freeze the lower ones and leave the top unfrozen
        # so it learns the specific microscopic atrophy of Alzheimer's.
        for param in list(self.base_model.features.parameters())[:-100]:
            param.requires_grad = False

        # 3. Extract the number of input features going into the final layer (1024 for DenseNet121)
        in_features = self.base_model.classifier.in_features

        # 4. Replace the default classification head with a robust Custom Head
        # Including: Linear (Dense), BatchNorm, ReLU, Dropout
        self.base_model.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4), # Prevents memorization of the training set

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),

            nn.Linear(256, num_classes)
            # PyTorch's CrossEntropyLoss calculates Softmax internally during training.
        )

        # 5. Isolated Softmax specifically for the Application / Prediction phase
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        """Standard Forward Pass for Training"""
        # DenseNet handles the Convolutions, MaxPooling, and Flattening internally
        logits = self.base_model(x)
        return logits

    def predict_probabilities(self, x):
        """Application Phase: Converts raw logits into 0-100% confidence percentages"""
        logits = self.forward(x)
        probabilities = self.softmax(logits)
        return probabilities

# Initialize the DenseNet model and send to GPU
model = DenseNetAlzheimer(num_classes=len(class_names)).to(device)
print("DenseNet-121 Architecture Initialized and moved to GPU.")

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 157MB/s]

DenseNet-121 Architecture Initialized and moved to GPU.


In [ ]:
# CELL 3: ACCELERATED TRAINING (AMP + OneCycleLR)
import torch.amp

NUM_EPOCHS = 15 # OneCycleLR reaches high accuracy quickly, 15 is plenty
SAVE_PATH = "best_densenet_alzheimer_44k.pth"

criterion = nn.CrossEntropyLoss()
# Crucial: Only optimize the unfrozen layers (Custom Head + Top 100 DenseNet layers)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

# 1. THE ONE-CYCLE SCHEDULER (Super Convergence)
steps_per_epoch = len(train_loader)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.003, # Spikes up to quickly escape local minima, then decays
    epochs=NUM_EPOCHS,
    steps_per_epoch=steps_per_epoch
)

# 2. AUTOMATIC MIXED PRECISION (AMP) SCALER (Doubles GPU speed)
scaler = torch.amp.GradScaler('cuda')

best_val_loss = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print(f"🚀 Commencing Accelerated DenseNet Training (AMP Enabled) for {NUM_EPOCHS} Epochs...")

for epoch in range(1, NUM_EPOCHS + 1):
    # --- TRAINING ---
    model.train()
    running_loss, running_corrects, total = 0.0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        # AMP Context Manager: Casts operations to 16-bit float for extreme speed
        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        # AMP Backward Pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # OneCycleLR steps per BATCH, not per epoch
        scheduler.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        running_corrects += torch.sum(preds == labels.data).item()
        total += labels.size(0)

    epoch_train_loss = running_loss / total
    epoch_train_acc = running_corrects / total

    # --- VALIDATION ---
    model.eval()
    val_loss, val_corrects, val_total = 0.0, 0, 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            # Use AMP for validation speed too
            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            val_corrects += torch.sum(preds == labels.data).item()
            val_total += labels.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_corrects / val_total

    # Get current LR for tracking
    current_lr = scheduler.get_last_lr()[0]

    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS:02d} | "
          f"LR: {current_lr:.6f} | "
          f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")

    # Save best model
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(best_model_wts, SAVE_PATH)

# Load best weights
model.load_state_dict(best_model_wts)
print("✅ Accelerated Training Complete. Best weights restored.")

/tmp/ipykernel_977/1827567224.py:21: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = torch.amp.GradScaler('cuda')


🚀 Commencing Accelerated DenseNet Training (AMP Enabled) for 15 Epochs...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/tmp/ipykernel_977/1827567224.py:41: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.amp.autocast('cuda'):


In [ ]:
# ==============================================================================
# NOVEL CBAM DUAL-ATTENTION ARCHITECTURE
# ==============================================================================
# 1. CUSTOM CHANNEL ATTENTION (The "What")
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc1 = nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        out = avg_out + max_out
        return self.sigmoid(out)

# 2. CUSTOM SPATIAL ATTENTION (The "Where")
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        assert kernel_size in (3, 7), 'kernel size must be 3 or 7'
        padding = 3 if kernel_size == 7 else 1
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv1(x)
        return self.sigmoid(x)

# 3. NOVEL CBAM DENSENET ARCHITECTURE
class OutOfTheBoxAlzheimerNet(nn.Module):
    def __init__(self, num_classes):
        super(OutOfTheBoxAlzheimerNet, self).__init__()

        weights = models.DenseNet121_Weights.DEFAULT
        self.base_model = models.densenet121(weights=weights)
        self.features = self.base_model.features

        # Freeze lower layers
        for param in list(self.features.parameters())[:-100]:
            param.requires_grad = False

        in_features = 1024

        # Inject custom Dual-Attention mechanism
        self.ca = ChannelAttention(in_features)
        self.sa = SpatialAttention()

        # Advanced classification head with Mish activation
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(in_features, 512),
            nn.Mish(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(p=0.4),
            nn.Linear(512, 128),
            nn.Mish(inplace=True),
            nn.BatchNorm1d(128),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes)
        )
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.features(x)
        # Apply Channel Attention, then Spatial Attention
        x = x * self.ca(x)
        x = x * self.sa(x)
        logits = self.classifier(x)
        return logits

    def predict_probabilities(self, x):
        logits = self.forward(x)
        return self.softmax(logits)

# Initialize and send to GPU
novel_model = OutOfTheBoxAlzheimerNet(num_classes=len(class_names)).to(device)
print("CBAM Dual-Attention Architecture Initialized.")

In [ ]:
# ==============================================================================
# NOVEL FOCAL LOSS TRAINING (AMP + OneCycleLR)
# ==============================================================================
# CUSTOM FOCAL LOSS
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(reduction='none')(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss.sum()

NUM_EPOCHS = 15
NOVEL_SAVE_PATH = "novel_cbam_focal_alzheimer.pth"

focal_criterion = FocalLoss(gamma=2.0)
novel_optimizer = optim.Adam(filter(lambda p: p.requires_grad, novel_model.parameters()), lr=0.001)

novel_scheduler = optim.lr_scheduler.OneCycleLR(
    novel_optimizer,
    max_lr=0.003,
    epochs=NUM_EPOCHS,
    steps_per_epoch=len(train_loader)
)

novel_scaler = torch.amp.GradScaler('cuda')

best_novel_val_loss = float('inf')
best_novel_wts = copy.deepcopy(novel_model.state_dict())

# Keeping a separate history dictionary for head-to-head comparison
novel_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print(f"🚀 Commencing Novel CBAM + Focal Loss Training for {NUM_EPOCHS} Epochs...")

for epoch in range(1, NUM_EPOCHS + 1):
    novel_model.train()
    running_loss, running_corrects, total = 0.0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        novel_optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = novel_model(inputs)
            loss = focal_criterion(outputs, labels)

        novel_scaler.scale(loss).backward()
        novel_scaler.step(novel_optimizer)
        novel_scaler.update()
        novel_scheduler.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        running_corrects += torch.sum(preds == labels.data).item()
        total += labels.size(0)

    epoch_train_loss = running_loss / total
    epoch_train_acc = running_corrects / total

    novel_model.eval()
    val_loss, val_corrects, val_total = 0.0, 0, 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            with torch.amp.autocast('cuda'):
                outputs = novel_model(inputs)
                loss = focal_criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            val_corrects += torch.sum(preds == labels.data).item()
            val_total += labels.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_corrects / val_total
    current_lr = novel_scheduler.get_last_lr()[0]

    novel_history['train_loss'].append(epoch_train_loss)
    novel_history['train_acc'].append(epoch_train_acc)
    novel_history['val_loss'].append(epoch_val_loss)
    novel_history['val_acc'].append(epoch_val_acc)

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS:02d} | LR: {current_lr:.6f} | "
          f"Train Loss (Focal): {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | "
          f"Val Loss (Focal): {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")

    if epoch_val_loss < best_novel_val_loss:
        best_novel_val_loss = epoch_val_loss
        best_novel_wts = copy.deepcopy(novel_model.state_dict())
        torch.save(best_novel_wts, NOVEL_SAVE_PATH)

novel_model.load_state_dict(best_novel_wts)
print("Novel Training Complete!!! Best weights restored.")

In [ ]:
# ==============================================================================
# VANILLA VS. NOVEL COMPARISON & ROC-AUC
# ==============================================================================
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from itertools import cycle

# ==========================================
# 1. HEAD-TO-HEAD ACCURACY COMPARISON
# ==========================================
plt.figure(figsize=(10, 6))

# Plot Vanilla Model (from Cell 3's history)
plt.plot(history['val_acc'], label='Vanilla DenseNet (Val Acc)', linestyle='--', color='#1f77b4', linewidth=2)

# Plot Novel Model (from Cell 6's novel_history)
plt.plot(novel_history['val_acc'], label='Novel CBAM + Focal (Val Acc)', linestyle='-', color='#2ca02c', linewidth=3)

plt.title('Head-to-Head: Vanilla vs Novel Architecture', fontsize=16)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Validation Accuracy', fontsize=12)
plt.legend(fontsize=12)
plt.grid(alpha=0.4)
plt.show()

# ==========================================
# 2. GENERATE ROC-AUC DATA FOR NOVEL MODEL
# ==========================================
novel_model.eval()
novel_probs = []
novel_labels = []

print("\nGenerating Probabilities for ROC Curve...")
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device, non_blocking=True)
        # Using the custom predict function from Cell 5
        probabilities = novel_model.predict_probabilities(inputs)

        novel_probs.extend(probabilities.cpu().numpy())
        novel_labels.extend(labels.numpy())

novel_probs = np.array(novel_probs)
novel_labels = np.array(novel_labels)

# ==========================================
# 3. PLOT MULTICLASS ROC CURVE
# ==========================================
# Binarize labels for Multiclass ROC calculation
n_classes = len(class_names)
binarized_labels = label_binarize(novel_labels, classes=[0, 1, 2, 3])
fpr, tpr, roc_auc = dict(), dict(), dict()

# Calculate ROC for each individual class
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(binarized_labels[:, i], novel_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC
plt.figure(figsize=(10, 8))
colors = cycle(['aqua', 'darkorange', 'cornflowerblue', 'red'])

for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f'{class_names[i]} (AUC = {roc_auc[i]:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2) # Diagonal random-guess baseline
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Novel CBAM Architecture', fontsize=14)
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# ==============================================================================
# CELL 8: ADVANCED DIAGNOSTIC CHARTS (AREA & LINE)
# ==============================================================================
import matplotlib.pyplot as plt
import numpy as np

epochs = np.arange(1, len(history['val_loss']) + 1)

fig, axes = plt.subplots(3, 1, figsize=(12, 18))
plt.subplots_adjust(hspace=0.4)

# ---------------------------------------------------------
# 1. LINE CHART: Vanilla vs Novel Validation Loss
# ---------------------------------------------------------
axes[0].plot(epochs, history['val_loss'], label='Vanilla Val Loss (Cross-Entropy)',
             linestyle='--', color='#d62728', linewidth=2.5)
axes[0].plot(epochs, novel_history['val_loss'], label='Novel Val Loss (Focal)',
             linestyle='-', color='#2ca02c', linewidth=2.5)
axes[0].set_title('Validation Loss Trajectory (Lower is Better)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epochs', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=11)

# ---------------------------------------------------------
# 2. AREA CHART: The Overfitting Margin (Accuracy Gap)
# ---------------------------------------------------------
# Calculates the gap between Training and Validation Accuracy
# A widening gap indicates memorization. A tight, stable gap indicates learning.
novel_train_acc = np.array(novel_history['train_acc'])
novel_val_acc = np.array(novel_history['val_acc'])

axes[1].plot(epochs, novel_train_acc, color='#1f77b4', label='Train Accuracy', linewidth=2)
axes[1].plot(epochs, novel_val_acc, color='#ff7f0e', label='Validation Accuracy', linewidth=2)

# Fill the area between the curves to visualize the margin
axes[1].fill_between(epochs, novel_train_acc, novel_val_acc,
                     where=(novel_train_acc >= novel_val_acc),
                     interpolate=True, color='#1f77b4', alpha=0.2, label='Overfitting Margin')
axes[1].fill_between(epochs, novel_train_acc, novel_val_acc,
                     where=(novel_val_acc > novel_train_acc),
                     interpolate=True, color='#ff7f0e', alpha=0.2, label='Generalization Bonus')

axes[1].set_title('Novel Architecture Overfitting Margin (Area Chart)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epochs', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=11)

# ---------------------------------------------------------
# 3. AREA CHART: Focal Loss Convergence
# ---------------------------------------------------------
novel_train_loss = np.array(novel_history['train_loss'])
novel_val_loss = np.array(novel_history['val_loss'])

axes[2].plot(epochs, novel_train_loss, color='purple', label='Train Focal Loss', linewidth=2)
axes[2].plot(epochs, novel_val_loss, color='teal', label='Validation Focal Loss', linewidth=2)

# Fill the area under the curves to show absolute volume of loss
axes[2].fill_between(epochs, novel_train_loss, color='purple', alpha=0.1)
axes[2].fill_between(epochs, novel_val_loss, color='teal', alpha=0.2)

axes[2].set_title('Focal Loss Convergence Area (Novel Model)', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Epochs', fontsize=12)
axes[2].set_ylabel('Focal Loss', fontsize=12)
axes[2].grid(alpha=0.3)
axes[2].legend(fontsize=11)

plt.show()

In [ ]:
# ==============================================================================
# CELL 9: REAL-WORLD TESTING & INFERENCE PIPELINE
# ==============================================================================
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import io

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not in Google Colab. Please manually set image paths.")

# 1. Define the exact same preprocessing used during validation
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict_new_mri(image_bytes, image_name):
    """Processes a new MRI image and outputs the diagnosis and confidence."""
    # Ensure model is in evaluation mode
    novel_model.eval()

    # Load and preprocess the image
    img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    input_tensor = inference_transform(img).unsqueeze(0).to(device)

    # Generate predictions
    with torch.no_grad():
        # Use our custom prediction function to get Softmax probabilities
        probabilities = novel_model.predict_probabilities(input_tensor)[0]
        confidence, predicted_idx = torch.max(probabilities, 0)

    predicted_class = class_names[predicted_idx.item()]
    probs_array = probabilities.cpu().numpy() * 100

    # ---------------------------------------------------------
    # Visualization: Image and Probability Bar Chart
    # ---------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Display the MRI
    axes[0].imshow(img, cmap='gray')
    axes[0].axis('off')
    axes[0].set_title(f"Uploaded Scan: {image_name}\nDiagnosis: {predicted_class}",
                      fontsize=14, fontweight='bold', color='darkred')

    # Display the confidence bar chart
    y_pos = np.arange(len(class_names))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

    bars = axes[1].barh(y_pos, probs_array, color=colors)
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(class_names, fontsize=12)
    axes[1].invert_yaxis()  # Read top-to-bottom
    axes[1].set_xlabel('Confidence (%)', fontsize=12)
    axes[1].set_title('Model Confidence by Stage', fontsize=14, fontweight='bold')
    axes[1].set_xlim([0, 105])

    # Add percentage text to bars
    for bar in bars:
        width = bar.get_width()
        axes[1].text(width + 2, bar.get_y() + bar.get_height()/2,
                     f'{width:.2f}%', va='center', fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.show()

# 2. Trigger the upload prompt (Colab only)
if IN_COLAB:
    print("Please upload one or more MRI images (JPG, PNG) to test the model.")
    uploaded = files.upload()

    for filename, data in uploaded.items():
        print(f"\nAnalyzing: {filename}...")
        predict_new_mri(data, filename)

In [ ]:
# ==============================================================================
# CELL 10: EXTRACT SAMPLE TEST FILES
# ==============================================================================
import os
import shutil
import random

# Create a folder for the test images
test_dir = './Test_Scans'
os.makedirs(test_dir, exist_ok=True)

print("Extracting random sample images for testing...")

# Pick one random image from each class
for cls in class_names:
    cls_path = os.path.join(DATA_DIR, cls)
    if os.path.exists(cls_path):
        all_files = os.listdir(cls_path)
        if all_files:
            random_img = random.choice(all_files)
            src = os.path.join(cls_path, random_img)
            # Rename for clarity
            dst = os.path.join(test_dir, f"Sample_{cls}.jpg")
            shutil.copy(src, dst)
            print(f"✅ Saved: {dst}")

print("\nFiles are ready! Check the folder icon on the left of Colab to download them.")

# # ==============================================================================
# # FORCE DOWNLOAD TO LOCAL COMPUTER
# # ==============================================================================
# from google.colab import files
# import os

# test_dir = './Test_Scans'

# print("Downloading files to your computer...")
# for filename in os.listdir(test_dir):
#     file_path = os.path.join(test_dir, filename)
#     files.download(file_path)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==============================================================================
# CELL 11: DEPLOY WEB APP INTERFACE (UPDATED FOR GRADIO 6.0)
# ==============================================================================
# Install Gradio silently
!pip install gradio -q

import gradio as gr
import torch
from torchvision import transforms
from PIL import Image

# 1. Prediction Function for the Web Interface
def predict_mri_interface(image):
    """Processes the image from the web UI and returns a dictionary of probabilities."""
    novel_model.eval()

    # Gradio passes a PIL image automatically
    input_tensor = inference_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            probabilities = novel_model.predict_probabilities(input_tensor)[0]

    # Format the output as a dictionary for Gradio's Label component
    probs_dict = {class_names[i]: float(probabilities[i]) for i in range(len(class_names))}
    return probs_dict

# 2. Build the Web App UI (Removed deprecated allow_flagging)
interface = gr.Interface(
    fn=predict_mri_interface,
    inputs=gr.Image(type="pil", label="Upload MRI Scan"),
    outputs=gr.Label(num_top_classes=4, label="AI Diagnosis Confidence"),
    title="🧠 Novel CBAM Alzheimer's Diagnostic AI",
    description=(
        "**Out-of-the-box Architecture Showcase:**\n\n"
        "Upload an MRI scan to test the custom Dual-Attention DenseNet model. "
        "This network utilizes a custom Spatial Attention mechanism and Focal Loss "
        "to accurately separate the 4 stages of Alzheimer's Disease."
    )
)

# 3. Launch the App (Theme passed to launch method per Gradio 6.0 rules)
print("Launching AI Application...")
interface.launch(share=True, debug=False, theme=gr.themes.Soft(primary_hue="blue", secondary_hue="green"))

In [ ]:
import shutil
import os

# Define the Drive folder path
drive_folder = '/content/drive/MyDrive/Alzheimer_MRI_Models'
os.makedirs(drive_folder, exist_ok=True)

# Copy the model files
novel_model_path = "novel_cbam_focal_alzheimer.pth"
vanilla_model_path = "best_densenet_alzheimer_44k.pth"

if os.path.exists(novel_model_path):
    shutil.copy(novel_model_path, drive_folder)
    print(f"✅ Novel Model saved to: {drive_folder}/{novel_model_path}")

if os.path.exists(vanilla_model_path):
    shutil.copy(vanilla_model_path, drive_folder)
    print(f"✅ Vanilla Model saved to: {drive_folder}/{vanilla_model_path}")

In [ ]:
# ==============================================================================
# FIND SAVED MODELS
# ==============================================================================
import glob

pth_files = glob.glob("*.pth")

print("Saved model files found in your current directory:")
if len(pth_files) == 0:
    print("❌ No models found. Your Colab runtime disconnected and wiped the files.")
else:
    for f in pth_files:
        print(f"✅ Found: {f}")